# MicroFinML - Data Preprocessing
## Phase 2: Clean, Transform, and Prepare Data for ML Models

**Steps:**
1. Load raw data
2. Feature engineering
3. Encode categorical variables
4. Scale numeric features
5. Train-Validation-Test split
6. Handle class imbalance with SMOTE
7. Save processed data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
import warnings

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

from src.data_preprocessing import (
    load_data, clean_data, get_data_summary,
    create_engineered_features, build_preprocessor,
    split_data, apply_smote, preprocess_pipeline,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET
)

print('Modules loaded successfully!')

## 1. Load and Clean Data

In [ ]:
df = load_data('../data/raw/Loan Default.csv')
df = clean_data(df)

summary = get_data_summary(df)
print(f"\nDataset shape: {summary['shape']}")
print(f"Missing values: {summary['missing_values']}")
print(f"Duplicates: {summary['duplicates']}")
print(f"Target distribution: {summary['target_distribution']}")
df.head()

## 2. Feature Engineering

In [ ]:
df_eng = create_engineered_features(df)

new_features = [c for c in df_eng.columns if c not in df.columns]
print(f'New features created ({len(new_features)}):')
for f in new_features:
    print(f'  - {f}: dtype={df_eng[f].dtype}')

print(f'\nDataset shape after engineering: {df_eng.shape}')
df_eng.head()

In [ ]:
# Visualize engineered features vs Default
eng_numeric = ['IncomeToLoanRatio', 'LoanToIncomeRatio', 'EstMonthlyPayment',
               'PaymentToIncomeRatio', 'EmploymentStability']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(eng_numeric):
    ax = axes[i]
    df_eng[df_eng['Default']==0][col].hist(ax=ax, bins=40, alpha=0.6, label='Repay', color='#2196F3', density=True)
    df_eng[df_eng['Default']==1][col].hist(ax=ax, bins=40, alpha=0.6, label='Default', color='#FF5722', density=True)
    ax.set_title(f'{col}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

axes[-1].set_visible(False)
plt.suptitle('Engineered Feature Distributions by Default Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/eda_plots/engineered_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Run Full Preprocessing Pipeline

In [ ]:
data = preprocess_pipeline(
    filepath='../data/raw/Loan Default.csv',
    use_engineered=True,
    use_smote=True,
    save_dir='../data/processed'
)

print(f"\n=== Processed Data Summary ===")
print(f"Training set: {data['X_train'].shape}")
print(f"Validation set: {data['X_val'].shape}")
print(f"Test set: {data['X_test'].shape}")
print(f"\nTotal features: {len(data['feature_names'])}")
print(f"Feature names: {data['feature_names'][:10]}...")

In [ ]:
# Verify class distribution after SMOTE
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

titles = ['Training (After SMOTE)', 'Validation', 'Test']
y_sets = [data['y_train'], data['y_val'], data['y_test']]

for ax, title, y in zip(axes, titles, y_sets):
    counts = y.value_counts()
    ax.bar(['Repay (0)', 'Default (1)'], counts.values, color=['#2196F3', '#FF5722'], edgecolor='black')
    ax.set_title(f'{title}\n(n={len(y):,})', fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

plt.suptitle('Target Distribution Across Splits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/eda_plots/split_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPreprocessing complete! Data saved to ../data/processed/')